# Data Reading

### Reading CSV Files 

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = (spark
      .read
      .format("csv")
      .option("header",True)
      .option("inferSchema",True)
      .load("/Volumes/learnspark/raw/spark_volume/raw_orders/orders.csv"))
df.display()

### Reading JSON Files 

In [0]:
df_json = (spark
           .read
           .format("JSON")
           .option("inferSchema",True)
           .load("/Volumes/learnspark/raw/spark_volume/raw_orders/orders.json")
           )

df_json.display()

### Reading Parquet

In [0]:
df_parquet = (spark.
              read
              .format("parquet")
              .load("/Volumes/learnspark/raw/spark_volume/raw_orders/part-00000-tid-orders.c000.snappy.parquet"))
df_parquet.display()

### Reading JDBC

In [0]:
# my_url = "jdbc:postgresql://localhost:5432/postgres"
# myconnection = {"user": "postgres", 
#                 "password": "postgres",
#                 "driver": "org.postgresql.Driver"}
# df = (spark
#       .read
#       .jdbc(url = my_url, table = "orders",properties= myconnection))

# # OR

# df = (
#     spark.read
#     .format("jdbc")
#     .option("url", my_url)
#     .option("dbtable", "orders")
#     .option("user", "postgres")
#     .option("password", "postgres")
#     .option("driver", "org.postgresql.Driver")
#     .load()
# )

# CORRUPT RECORDS MODES

### Permissive 
- reads the complete file, if any corrupt records are they it will be stored in seperate column 

In [0]:
df_json = (spark
           .read
           .format("JSON")
           .option("inferSchema",True)
           .option("mode","PREMISSIVE")
           .load("/Volumes/learnspark/raw/spark_volume/raw_orders/orders.json")
           )

df_json.display()

### DROPMALFORMED 
- it will simple drop malformed record while reading

In [0]:
df_json = (spark
           .read
           .format("JSON")
           .option("inferSchema",True)
           .option("mode","DROPMALFORMED")
           .load("/Volumes/learnspark/raw/spark_volume/raw_orders/orders.json")
           )

df_json.display()

### FAILFAST
- if your downstream application is too sensitive that you cannot take risk, then you can use mode = FAILFAST, so that pipeline will fail immediately 

In [0]:
df_json = (spark
           .read
           .format("JSON")
           .option("inferSchema",True)
           .option("mode","FAILFAST")
           .load("/Volumes/learnspark/raw/spark_volume/raw_orders/orders.json")
           )

df_json.display()

# DATA SCHEMA

### StructType Method

In [0]:
df.schema

In [0]:
my_custom_schme = StructType([StructField('order_id', StringType(), True), StructField('customer_id', StringType(), True), StructField('order_date', DateType(), True), StructField('product_id', StringType(), True), StructField('quantity', IntegerType(), True), StructField('price', DoubleType(), True), StructField('order_status', StringType(), True), StructField('shipping_address', StringType(), True), StructField('city', StringType(), True), StructField('country', StringType(), True), StructField('payment_method', StringType(), True), StructField('discount', DoubleType(), True), StructField('category', StringType(), True), StructField('sales_rep', StringType(), True), StructField('region', StringType(), True), StructField('ship_date', DateType(), True), StructField('delivery_days', IntegerType(), True), StructField('returned', StringType(), True), StructField('gender', StringType(), True)])

In [0]:
df_csv = (spark
          .read
          .format("csv")
          .option("header",True)
          .schema(my_custom_schme)
          .load("/Volumes/learnspark/raw/spark_volume/raw_orders/orders.csv"))

df_csv.display()

### DDL Schema

In [0]:
my_ddl_schema = """
order_id INT,
customer_id STRING,
order_date DATE,
product_id STRING,
quantity INTEGER,
price DOUBLE,
order_status STRING,
shipping_address STRING,
city STRING,
country STRING,
payment_method STRING,
discount DOUBLE,
category STRING,
sales_rep STRING,
region STRING,
ship_date DATE,
delivery_days INTEGER,
returned STRING,
gender STRING
"""
df_csv = (spark
          .read
          .format("csv")
          .option("header",True)
          .schema(my_ddl_schema)
          .load("/Volumes/learnspark/raw/spark_volume/raw_orders/orders.csv")
         )
display(df_csv)

### SELECT

In [0]:
df_select = df_csv.select("city","country","category")
# OR
df_select = df_csv.select(col("city"),col("country"),col("category"))

df_select.display()

### Alias

In [0]:
df_alias = df_csv.select(col("city").alias("cutomer_city"),col("country").alias("customer_country"),"category")
df_alias.display()

### Filter

#### Scenario_1

In [0]:
display(df_csv.filter(
    col("order_status") == "Returned"
    ))


#### scenario_2

In [0]:
display(df_csv.filter(
    (col("order_status") == "Returned") |
    (col("order_status") == "Cancelled")
    ))

#### isin

In [0]:
desired_order = ["Returned","Cancelled","Shipped"]
display(df_csv.filter(
    col("order_status").isin("Returned","Cancelled","Shipped")
    ))
# OR
display(df_csv.filter(
    col("order_status").isin(desired_order)
    ))

#### withColumnRenamed

In [0]:
df.withColumnRenamed("order_status","status").display()

#### withColumn
- This is the go-to API for either transforming the column or adding/creating a new one

##### scenario_1

In [0]:
df.withColumn("file_path",col("_metadata.file_path")).withColumn('flag',lit('0')).display()

##### scenario-2

In [0]:
display(
    df.withColumn('shipping_address',regexp_replace("shipping_address",',.*',''))
)

##### scenario-3

In [0]:
display(
    df.withColumn("Total_Price",round(col("price")*col("quantity"),2))
)
# OR
display(
    df.withColumn("Total_Price",col("price")*col("quantity")).withColumn("Total_Price",round("Total_Price",2))
)

### TypeCasting

In [0]:
display(df.withColumn("order_id",col("order_id").cast(StringType())))
# OR
display(df.withColumn("order_id",col("order_id").cast("STRING")))

### Sorting

#### scenario-1

In [0]:
df.sort(col("order_date").desc()).display()

#### scenario-2

In [0]:
display(df.sort(["order_date","quantity"],ascending=[0,1]))

### Limit

In [0]:
display(df.limit(10))

In [0]:
df_drop = df.withColumn("Total_Price",round(col("price")*col("quantity"),2))

### DROP

In [0]:
display(df_drop)
df_drop = df_drop.drop("Total_Price")
display(df_drop)

### dropDuplicates

### scenario-1

In [0]:
df_dedups = df.dropDuplicates()
display(df_dedups)

### scenario-2

In [0]:
df_dedups2 = df.dropDuplicates(subset=["order_date","product_id"])
display(df_dedups2)

### union & unionByName

In [0]:
df_union = df_dedups2.union(df_csv)
display(df_union)

In [0]:
df_change_col_order = df_csv.select("order_id","category",col("order_date"),"product_id","quantity","price","order_status","shipping_address","city","country","payment_method","discount","customer_id","sales_rep","region","ship_date","delivery_days","returned","gender"
)

In [0]:
df_union = df_dedups2.union(df_change_col_order)
display(df_union)

In [0]:
df_unionByName = df_dedups2.unionByName(df_change_col_order)
display(df_union)

## Date Functions

In [0]:
df_curr = df.withColumn("current_time",current_timestamp())
display(df_curr)

In [0]:
df_add = df_curr.withColumn("current_time",date_add("current_time",7))
display(df_add)

In [0]:
df_sub = df_curr.withColumn("current_time",date_sub("current_time",7))
display(df_sub)

In [0]:
df_diff = df_curr.withColumn("duration",datediff("current_time","ship_date"))
display(df_diff)

In [0]:
df_fortmat = df_curr.withColumn("current_time",date_format("current_time","yyyy-MM-dd"))
df_fortmat.display()

### STRING FUNCTION

In [0]:
df_upper = df_curr.withColumn("order_status",upper(col("order_status")))
df_upper.display()

In [0]:
df_str = df_curr.withColumn("status_lenght",length("order_status"))
df_str.display()

### Handling Nulls

In [0]:
%py
df_customers = spark.sql("""select * from gizmobox.bronze.v_customers""")

In [0]:
df_customers.display()

In [0]:
df_all_nulls = df_customers.dropna('all')
display(df_all_nulls)

In [0]:
df_any_null = df_customers.dropna('any') # by default dropna will take parameter as 'any'
display(df_any_null)

In [0]:
df_subset_null = df_customers.dropna(subset = ["customer_id","email"])
display(df_subset_null)

In [0]:
df_subset_null = df_customers.dropna(subset = ["customer_id","email"],how = 'all') # by default how = 'any'
display(df_subset_null)

In [0]:
df_fillna = df_customers.fillna('dummy')
display(df_fillna)


In [0]:
df_custom_fillna = df_customers.fillna({'email':'dummy@outlook.com','telephone':'+91 9492751026'})
display(df_custom_fillna)


### Split and Indexing

In [0]:
df_split = df_csv.withColumn("street_address",split("shipping_address",",")[0])\
    .withColumn("city_name",split("shipping_address",",")[1])
display(df_split)

### Explode
- it is used to apply explode in the arrays, Exploded in the rows(explode will not keep null, if you need null values to be present use explode_outer)

In [0]:
df_split = df_csv.withColumn("address_array",split("shipping_address",","))
df_explode = df_split.withColumn("address_explode",explode("address_array"))
display(df_explode)

In [0]:
df_split = df_csv.withColumn("address_array",split("shipping_address",","))
df_explode = df_split.withColumn("address_explode",explode_outer("address_array"))
display(df_explode)

### Array Contains

In [0]:
df_arr_contains = df_csv.withColumn("address_array",split("shipping_address",","))\
    .withColumn("city_1_flag",array_contains("address_array"," City1")).select("order_id","address_array","city_1_flag")
display(df_arr_contains)

### Group By

In [0]:
df_agg = (
    df_csv
    .withColumn("total_price",col("price")*col("quantity"))
    .groupBy("product_id")
    .agg(sum("total_price").alias("total_price")).withColumn("total_price",round(col("total_price"),2)).sort(col("total_price").desc())
    .select("product_id","total_price")
    )
display(df_agg)

In [0]:
af_agg_new = (
    df_csv.withColumn("total_sales",round(col("price")*col("quantity"),2))
    .groupBy("product_id","customer_id")
    .agg(sum("total_sales").alias("total_price"),avg("total_sales").alias("avg_price"))
    .withColumn("total_price",round(col("total_price"),2))
    .withColumn("avg_price",round(col("avg_price"),2))
    .sort("product_id","total_price",ascending = [True,False])
)
display(af_agg_new)

### Approx count

In [0]:
df = df_csv.groupBy("product_id").agg(approx_count_distinct("customer_id").alias("distinct_customers"))
display(df)

In [0]:
df_count = df_csv.groupBy("product_id").agg(countDistinct("customer_id").alias("distinct_customers"))
display(df_count)

### Collect_List

In [0]:
df_collect = df_csv.groupBy("customer_id").agg(collect_list('product_id').alias('products'))
display(df_collect)

In [0]:
df_collect = df_csv.groupBy("customer_id").agg(collect_set('product_id').alias('products'))
display(df_collect)

### Pivoting

In [0]:
df_pivot = df_csv.groupBy("customer_id").pivot("order_status").agg(count("order_id"))
display(df_pivot)

In [0]:
df_group = df_csv.groupBy("customer_id","order_status").agg(count("order_status").alias("count_status")).sort("customer_id")
display(df_group)

### when-otherwise

In [0]:
df_when = (df_pivot
           .withColumn("Return_Flag",when(col("Returned") == 1, "low").when(col("Returned") <= 3, "Medium")
                       .when(col("Returned") > 3, "High")
                       .otherwise("no returns"))
)
display(df_when)

### JOINS

#### Inner Join
- When you try to fetch only common data in both tables 

In [0]:
data = [
    (1,),
    (0,),
    (1,),
    (None,)
]

columns = ["id"]

df1 = spark.createDataFrame(data,columns)
df1.display()

In [0]:
data = [
    (1,),
    (None,),
    (0,),
    (None,)
]

columns = ["id"]

df2 = spark.createDataFrame(data,columns)
df2.display()

In [0]:
df_inner = df1.join(df2,df1["id"] == df2["id"],"inner")
display(df_inner)

#### Left Join
1. Inner + rest of the data from left table 

In [0]:
df_left = df1.join(df2,df1["id"] == df2["id"],"left")
display(df_left)

#### Right Join
1. inner + rest of the data from right table 

In [0]:
df_right = df1.join(df2,df1["id"] == df2["id"],"right")
display(df_right)

#### Full Join
1. Inner join + rest of the data from both tables 

In [0]:
df_full = df1.join(df2,df1["id"] == df2["id"],"full")
display(df_full)

#### Anti join

In [0]:
df_anti = df1.join(df2,df1["id"] == df2["id"],"anti")
display(df_anti)

In [0]:
df_semi = df1.join(df2,df1["id"] == df2["id"],"semi")
display(df_semi)

### Window Functions

### Row_number - Unique numbers for each records

In [0]:
query = """WITH employee AS (
    SELECT 101 AS empid, 10 AS deptid, 'Ajay' AS emp_name, 75000 AS salary
    UNION ALL
    SELECT 102, 10, 'Rahul', 68000
    UNION ALL
    SELECT 103, 20, 'Priya', 82000
    UNION ALL
    SELECT 104, 20, 'Sneha', 79000
    UNION ALL
    SELECT 105, 30, 'Vikram', 90000
    UNION ALL
    SELECT 106, 30, 'Kiran', 72000
    UNION ALL
    SELECT 107, 40, 'Meena', 65000
    UNION ALL
    SELECT 108, 40, 'Arun', 61000
    UNION ALL
    SELECT 109, 50, 'Divya', 88000
    UNION ALL
    SELECT 110, NULL, 'Ramesh', 55000
)
select * from employee"""

df_employee = spark.sql(query)

In [0]:
query = """WITH department AS (
    SELECT 10 AS dept_id, 'Engineering' AS dept_name, 'Suresh' AS hod
    UNION ALL
    SELECT 20, 'Finance', 'Lakshmi'
    UNION ALL
    SELECT 30, 'Human Resources', 'Anita'
    UNION ALL
    SELECT 40, 'Sales', 'Mahesh'
    UNION ALL
    SELECT 50, 'Analytics', 'Ravi'
    UNION ALL
    SELECT 60, 'Legal', 'Kavitha'
)
select * from  department"""

df_department = spark.sql(query)

In [0]:
from pyspark.sql.window import Window
df_row = df_employee.withColumn("rank",row_number().over(Window.partitionBy("deptid").orderBy(df_employee.deptid)))
display(df_row)

### Rank VS Dense_Rank

In [0]:
from pyspark.sql.window import Window
df_rank = df_employee.withColumn("rank",rank().over(Window.orderBy(df_employee.deptid)))\
    .withColumn("dense_rank",dense_rank().over(Window.orderBy(df_employee.deptid)))
display(df_rank)

### Special Window function or Special Functions

In [0]:
data = [(2020,100),(2021,110),(2023,90),(2022,140),(2024,150)]
df = spark.createDataFrame(data,"Year INT,revenue INT")
display(df)

In [0]:
df_total = df.withColumn("total_revenue",sum("revenue").over(Window.orderBy("year")))
display(df_total)

In [0]:
df_total = df.withColumn("total_revenue",sum("revenue").over(Window.orderBy("year").rowsBetween(Window.unboundedPreceding,Window.currentRow)))
display(df_total) # default Window.unboundedPreceding,Window.currentRow

In [0]:
df_total = df.withColumn("total_revenue",sum("revenue").over(Window.orderBy("year").rowsBetween(Window.unboundedPreceding,Window.unboundedFollowing)))
display(df_total) # default Window.unboundedPreceding,Window.unboundedFollowing

### UDF - User Defined Functions 

In [0]:
def square(x):
    return x*x
square(20.1)

In [0]:
my_square = udf(square)

In [0]:
df_cust = df_csv.withColumn("price_square",my_square("price"))
display(df_cust)

In [0]:
@udf(returnType= FloatType())
def square(x):
    return x*x

In [0]:
df_cust = df_csv.withColumn("price_square",square("price"))
display(df_cust)

### User Defined Table Functions(UDTF) 
- returns set of rows or dataframe where as udf returns single value.
- For UDTF, we need to create a clas not function.
- Function name should be used as eval

In [0]:
df_demo = spark.createDataFrame([("Hello World",),("just an Example",),("Hi Bro",)],["text"])
display(df_demo)

In [0]:
@udtf(returnType="word STRING")
class udtf_class:
    def eval(self,text:str): # Function name to be used
        for  word in text.split():
            yield (word,)

In [0]:
udtf_class(lit("Hello World, This is a developer function")).show()

### Call UDF

In [0]:
@udf(returnType= FloatType())
def square(x):
    return x*x

In [0]:
spark.udf.register("square",square)
df_cust = df_csv.withColumn("price_square",call_udf("square", "price"))
display(df_cust)

### concat & concat_ws

In [0]:
df_con = df_csv.withColumn("custom_id",concat("customer_id",lit("|"),"product_id",lit("|"),"order_id"))
display(df_con)

In [0]:
df_conws = df_csv.withColumn("custom_id",concat_ws(lit('|'),"customer_id","product_id","order_id"))
display(df_con)

### Data Writing

In [0]:
(df_csv
 .write
 .format("csv")
 .mode("overwrite")
 .option("path","/Volumes/learnspark/raw/spark_volume/destination/csv")
 .save()
 )

In [0]:
(df_csv
 .write
 .format("csv")
 .mode("append")
 .option("path","/Volumes/learnspark/raw/spark_volume/destination/csv")
 .save()
 )

In [0]:
(df_csv
 .write
 .format("csv")
 .mode("error")
 .option("path","/Volumes/learnspark/raw/spark_volume/destination/csv")
 .save()
 )

In [0]:
(df_csv
 .write
 .format("csv")
 .mode("ignore")
 .option("path","/Volumes/learnspark/raw/spark_volume/destination/csv")
 .save()
 )

### File Formats

#### csv

In [0]:
(df_csv
 .write
 .format("csv")
 .mode("append")
 .option("path","/Volumes/learnspark/raw/spark_volume/destination/csv")
 .save()
 )

### JSON

In [0]:
(df_csv
 .write
 .format("json")
 .mode("append")
 .option("path","/Volumes/learnspark/raw/spark_volume/destination/json")
 .save()
 )

In [0]:
(df_csv
 .write
 .format("parquet")
 .mode("append")
 .option("path","/Volumes/learnspark/raw/spark_volume/destination/parquet")
 .save()
 )

### Delta Lake or Delta Format

In [0]:
(df_csv
 .write
 .format("delta")
 .mode("append")
 .option("path","/Volumes/learnspark/raw/spark_volume/destination/delta")
 .save()
 )

### Upsert with Delta Library

In [0]:
df_new = df_csv.filter(col('order_id').isin(1001,1002))
df_new_1 = df_new.filter(col("order_id") == 1001).withColumn("product_id",lit('P102'))
df_new_2 = df_new.filter(col("order_id") == 1002).withColumn("order_id",lit('90001'))
df_new = df_new_1.union(df_new_2)
display(df_new)

In [0]:
from delta.tables import DeltaTable
dlt_object = DeltaTable.forPath(spark,"/Volumes/learnspark/raw/spark_volume/destination/delta/")
dlt_object.alias("trg").merge(df_new.alias("src"),"trg.order_id = src.order_id")\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()

In [0]:
df_test = spark.read.format("delta").load("/Volumes/learnspark/raw/spark_volume/destination/delta/")
display(df_test)

### Handling JSON

In [0]:
df_json = (spark
           .read
           .format("JSON")
           .option("multiLine",True)
           .option("inferSchema",True)
           .load("/Volumes/learnspark/raw/spark_volume/json_orders/")
           )
# Explode Items
df_json = df_json.withColumn("items",explode(col("items")))

# fetch cols from items 
df_json = df_json.withColumn("product_id",col("items.item_id"))\
    .withColumn("price",col("items.price"))\
        .withColumn("product_name",col("items.product_name"))\
            .withColumn("quantity",col("items.quantity"))\
            .drop("items")\
                .withColumn("customer_id",col("customer.customer_id"))\
                    .withColumn("email",col("customer.email"))\
                        .withColumn("customer_name",col("customer.name"))\
                            .withColumn("city",col("customer.address.city"))\
                            .drop("customer")\
                                .withColumn("payment_method",col("payment.method"))\
                                    .withColumn("transaction_id",col("payment.transaction_id"))\
                                        .drop("payment","metadata")

display(df_json)

### File Source Options
[Generic File Options](https://spark.apache.org/docs/latest/sql-data-sources-generic-options.html)